In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.model_selection import train_test_split

In [ ]:
df_fake = pd.read_csv('fake.csv')
df_true = pd.read_csv('true.csv')

In [ ]:
df_fake['label'] = 0
df_true['label'] = 1

In [ ]:
df = pd.concat([df_fake, df_true],axis = 0).reset_index(drop = True)

In [ ]:
print(df['label'].value_counts())

In [ ]:
df.head()

In [ ]:
df['text'] = df['text'].str.lower()
df['text'] = df['text'].apply(lambda x: re.sub(r'[^\w\s]','',x))
nltk.download('stopwords')
stopwords_list = set(stopwords.words('english'))
df['text'] = df['text'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stopwords_list)]))
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
df['text'] = df['text'].apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))

In [ ]:
print(df['text'])

In [ ]:
X = df['text']
Y = df['label']
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size = 0.2,random_state = 42)
tfid = TfidfVectorizer(max_features = 5000)
X_tfid_train = tfid.fit_transform(X_train)
Y_tfid_test = tfid.transform(X_test)
countvect = CountVectorizer(max_features = 5000)
X_countvect_train = countvect.fit_transform(X_train)
Y_countvect_test = countvect.transform(X_test)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,f1_score

In [ ]:
svm_count = LinearSVC()
svm_count.fit(X_countvect_train,Y_train)
y_pred_count = svm_count.predict(Y_countvect_test)

In [ ]:
print(accuracy_score(Y_test,y_pred_count))
print(confusion_matrix(Y_test,y_pred_count))
print(classification_report(Y_test,y_pred_count))
print(f1_score(Y_test,y_pred_count))

In [ ]:
svm_tfid = LinearSVC()
svm_tfid.fit(X_tfid_train,Y_train)
y_pred_tfid = svm_tfid.predict(Y_tfid_test)

In [ ]:
print(accuracy_score(Y_test,y_pred_tfid))
print(confusion_matrix(Y_test,y_pred_tfid))
print(classification_report(Y_test,y_pred_tfid))
print(f1_score(Y_test,y_pred_tfid))

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

In [ ]:
fake_text = " ".join(df[df['label'] == 0]['text'])
real_text = " ".join(df[df['label'] == 1]['text'])

fake_wc = WordCloud(width=800, height=400, background_color='black', colormap='Reds').generate(fake_text)
real_wc = WordCloud(width=800, height=400, background_color='black', colormap='Blues').generate(real_text)

plt.figure(figsize=(14,6))

plt.subplot(1,2,1)
plt.imshow(fake_wc, interpolation="bilinear")
plt.axis("off")
plt.title("Fake News - Common Words", fontsize=16)

plt.subplot(1,2,2)
plt.imshow(real_wc, interpolation="bilinear")
plt.axis("off")
plt.title("Real News - Common Words", fontsize=16)

plt.show()